Create manual or reproducible random input data.

In [10]:
from __future__ import annotations
import random
def parse_preference_entry(entry: str, hospitals: list[str]) -> list[str]:
    tokens = entry.replace(",", " ").split()
    converted = []
    for token in tokens:
        if token.isdigit():
            index = int(token) - 1
            if index < 0 or index >= len(hospitals):
                raise ValueError(f"Hospital number {token} is outside the valid range.")
            converted.append(hospitals[index])
        else:
            converted.append(token)

    if len(converted) != len(hospitals):
        raise ValueError(f"Enter exactly {len(hospitals)} hospitals.")
    if len(set(converted)) != len(converted):
        raise ValueError("Each hospital must appear exactly once.")
    if set(converted) != set(hospitals):
        raise ValueError("Preferences must contain every hospital exactly once.")
    return converted

def generate_random_preferences(
    doctors: list[str],
    hospitals: list[str],
    seed: int | None = None,
) -> dict[str, list[str]]:
    generator = random.Random(seed)
    return {
        doctor: generator.sample(hospitals, len(hospitals))
        for doctor in doctors
    }
def collect_manual_preferences(
    doctors: list[str],
    hospitals: list[str],
    input_function=input,
    output_function=print,
) -> dict[str, list[str]]:
    output_function("Hospitals: " + ", ".join(
        f"{index}={hospital}" for index, hospital in enumerate(hospitals, start=1)
    ))
    preferences = {}
    for doctor in doctors:
        while True:
            entry = input_function(
                f"Preferences for {doctor} (best to worst, separated by spaces): "
            )
            try:
                preferences[doctor] = parse_preference_entry(entry, hospitals)
                break
            except ValueError as error:
                output_function(f"Invalid preference list: {error}")
    return preferences

Input data

In [11]:
def read_positive_integer(prompt: str) -> int:
    """Read a positive integer, repeating until the input is valid."""
    while True:
        try:
            value = int(input(prompt))
            if value <= 0:
                raise ValueError
            return value
        except ValueError:
            print("Please enter a positive integer.")

def read_capacities(hospitals: list[str], doctor_count: int) -> dict[str, int]:
    """Read capacities and repeat the complete entry if total capacity is low."""
    while True:
        capacities = {}
        print("\nEnter a non-negative integer capacity for each hospital.")
        for hospital in hospitals:
            while True:
                try:
                    capacity = int(input(f"Capacity for {hospital}: "))
                    if capacity < 0:
                        raise ValueError
                    capacities[hospital] = capacity
                    break
                except ValueError:
                    print("Capacity must be a non-negative integer.")

        total_capacity = sum(capacities.values())
        if total_capacity >= doctor_count:
            return capacities
        print(
            f"Total capacity is {total_capacity}, but {doctor_count} doctors "
            "need assignments. Please enter all capacities again."
        )
def collect_preferences(
    doctors: list[str], hospitals: list[str]
) -> dict[str, list[str]]:
    """Let the user select manual or reproducible random preferences."""
    while True:
        mode = input("\nPreference mode: manual (m) or random demo (r)? ").strip().lower()
        if mode in {"m", "manual"}:
            return collect_manual_preferences(doctors, hospitals)
        if mode in {"r", "random"}:
            seed_entry = input("Random seed [0]: ").strip()
            try:
                seed = int(seed_entry) if seed_entry else 0
            except ValueError:
                print("Random seed must be an integer.")
                continue
            return generate_random_preferences(doctors, hospitals, seed=seed)
        print("Enter m for manual input or r for a random demo.")
def print_preferences(preferences: dict[str, list[str]]) -> None:
    """Display complete preference rankings."""
    print("\nPreferences")
    print("-" * 50)
    for doctor, hospitals in preferences.items():
        print(f"{doctor}: {' > '.join(hospitals)}")
def print_assignment(title: str, assignment: dict[str, dict[str, str]]) -> None:
    """Display one assignment in doctor order."""
    print(f"\n{title}")
    print("-" * 50)
    for doctor, result in assignment.items():
        print(f"{doctor} -> {result['hospital']}")
def collect_inputs():
    doctor_count = read_positive_integer("Number of doctors: ")
    hospital_count = read_positive_integer("Number of hospitals: ")

    doctors = [f"D{index}" for index in range(1, doctor_count + 1)]
    hospitals = [f"H{index}" for index in range(1, hospital_count + 1)]

    capacities = read_capacities(hospitals, doctor_count)
    preferences = collect_preferences(doctors, hospitals)

    print_preferences(preferences)

    return doctors, hospitals, capacities, preferences


doctors, hospitals, capacities, preferences = collect_inputs()



Enter a non-negative integer capacity for each hospital.

Preferences
--------------------------------------------------
D1: H3 > H2 > H5 > H1 > H4 > H6 > H8 > H10 > H9 > H7
D2: H2 > H8 > H1 > H7 > H4 > H6 > H9 > H3 > H10 > H5
D3: H4 > H2 > H6 > H1 > H7 > H8 > H5 > H9 > H10 > H3
D4: H1 > H9 > H4 > H7 > H8 > H6 > H2 > H5 > H10 > H3
D5: H8 > H5 > H1 > H4 > H9 > H10 > H2 > H3 > H7 > H6
D6: H6 > H9 > H7 > H5 > H10 > H2 > H3 > H8 > H4 > H1
D7: H10 > H1 > H8 > H2 > H6 > H4 > H5 > H3 > H9 > H7
D8: H9 > H6 > H2 > H4 > H10 > H5 > H1 > H7 > H8 > H3
D9: H8 > H1 > H10 > H9 > H3 > H5 > H4 > H6 > H7 > H2
D10: H9 > H4 > H1 > H7 > H2 > H5 > H6 > H10 > H3 > H8
D11: H8 > H5 > H1 > H4 > H6 > H9 > H2 > H3 > H10 > H7
D12: H1 > H8 > H6 > H5 > H7 > H2 > H4 > H9 > H3 > H10
D13: H6 > H1 > H10 > H4 > H5 > H9 > H2 > H3 > H8 > H7
D14: H2 > H9 > H5 > H1 > H6 > H7 > H8 > H4 > H10 > H3
D15: H5 > H4 > H10 > H1 > H8 > H2 > H3 > H6 > H7 > H9
D16: H3 > H5 > H10 > H6 > H8 > H7 > H4 > H9 > H2 > H1
D17: H8 > H2 > H1 > H3 

Input validation for the doctor-hospital assignment problem.

In [12]:
def validate_inputs(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
) -> None:
    if not isinstance(preferences, dict):
        raise TypeError("Preferences must be a dictionary.")
    if not isinstance(capacities, dict):
        raise TypeError("Capacities must be a dictionary.")
    if not preferences:
        raise ValueError("At least one doctor must be provided.")
    if not capacities:
        raise ValueError("At least one hospital must be provided.")

    hospital_names = set(capacities)
    for hospital, capacity in capacities.items():
        if not isinstance(hospital, str) or not hospital.strip():
            raise ValueError("Every hospital name must be a non-empty string.")
        if isinstance(capacity, bool) or not isinstance(capacity, int):
            raise TypeError(f"Capacity for {hospital!r} must be an integer.")
        if capacity < 0:
            raise ValueError(f"Capacity for {hospital!r} cannot be negative.")

    for doctor, ranked_hospitals in preferences.items():
        if not isinstance(doctor, str) or not doctor.strip():
            raise ValueError("Every doctor name must be a non-empty string.")
        if not isinstance(ranked_hospitals, list):
            raise TypeError(f"Preferences for {doctor!r} must be a list.")
        if len(ranked_hospitals) != len(capacities):
            raise ValueError(f"{doctor!r} must rank every hospital exactly once.")
        if any(not isinstance(name, str) or not name.strip() for name in ranked_hospitals):
            raise ValueError(f"All hospitals ranked by {doctor!r} must have valid names.")
        if len(set(ranked_hospitals)) != len(ranked_hospitals):
            raise ValueError(f"Preferences for {doctor!r} contain duplicate hospitals.")
        if set(ranked_hospitals) != hospital_names:
            missing = sorted(hospital_names - set(ranked_hospitals))
            unknown = sorted(set(ranked_hospitals) - hospital_names)
            raise ValueError(
                f"Preferences for {doctor!r} do not match the hospital list. "
                f"Missing: {missing}; unknown: {unknown}."
            )

    total_capacity = sum(capacities.values())
    if total_capacity < len(preferences):
        raise ValueError(
            "Total hospital capacity must be at least the number of doctors. "
            f"Doctors: {len(preferences)}; total capacity: {total_capacity}."
        )

Hungarian assignment solver with hospital capacity slots.

In [13]:
def create_slots(capacities: dict[str, int]) -> list[dict[str, str]]:
    """Expand each hospital capacity into one-to-one assignment slots."""
    return [
        {"hospital": hospital, "slot": f"{hospital}_{slot_number}"}
        for hospital, capacity in capacities.items()
        for slot_number in range(1, capacity + 1)
    ]


def create_rank_lookup(
    preferences: dict[str, list[str]],
) -> dict[str, dict[str, int]]:
    """Convert ordered preference lists to one-based rank costs."""
    return {
        doctor: {
            hospital: rank
            for rank, hospital in enumerate(preference_list, start=1)
        }
        for doctor, preference_list in preferences.items()
    }


def build_cost_matrix(
    doctors: list[str],
    slots: list[dict[str, str]],
    rank_lookup: dict[str, dict[str, int]],
) -> list[list[int]]:
    """Build a doctor-by-slot rank cost matrix."""
    return [
        [rank_lookup[doctor][slot["hospital"]] for slot in slots]
        for doctor in doctors
    ]


def hungarian_algorithm(cost_matrix: list[list[int]]) -> list[int]:
    """Return the minimum-cost slot index for every matrix row."""
    if not cost_matrix or not cost_matrix[0]:
        raise ValueError("Cost matrix cannot be empty.")
    if any(len(row) != len(cost_matrix[0]) for row in cost_matrix):
        raise ValueError("Cost matrix must be rectangular.")

    row_count = len(cost_matrix)
    column_count = len(cost_matrix[0])
    if row_count > column_count:
        raise ValueError(
            "Number of hospital slots must be at least the number of doctors."
        )

    row_potential = [0] * (row_count + 1)
    column_potential = [0] * (column_count + 1)
    matched_row = [0] * (column_count + 1)
    previous_column = [0] * (column_count + 1)

    for row in range(1, row_count + 1):
        matched_row[0] = row
        current_column = 0
        minimum_reduced_cost = [float("inf")] * (column_count + 1)
        used = [False] * (column_count + 1)

        while True:
            used[current_column] = True
            current_row = matched_row[current_column]
            delta = float("inf")
            next_column = 0

            for column in range(1, column_count + 1):
                if not used[column]:
                    reduced_cost = (
                        cost_matrix[current_row - 1][column - 1]
                        - row_potential[current_row]
                        - column_potential[column]
                    )
                    if reduced_cost < minimum_reduced_cost[column]:
                        minimum_reduced_cost[column] = reduced_cost
                        previous_column[column] = current_column
                    if minimum_reduced_cost[column] < delta:
                        delta = minimum_reduced_cost[column]
                        next_column = column

            for column in range(column_count + 1):
                if used[column]:
                    row_potential[matched_row[column]] += delta
                    column_potential[column] -= delta
                else:
                    minimum_reduced_cost[column] -= delta

            current_column = next_column
            if matched_row[current_column] == 0:
                break

        while True:
            next_column = previous_column[current_column]
            matched_row[current_column] = matched_row[next_column]
            current_column = next_column
            if current_column == 0:
                break

    assignment = [-1] * row_count
    for column in range(1, column_count + 1):
        if matched_row[column] != 0:
            assignment[matched_row[column] - 1] = column - 1
    return assignment


def solve_hungarian(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
) -> dict[str, dict[str, str]]:
    """Validate and solve a complete doctor-hospital assignment problem."""
    validate_inputs(preferences, capacities)
    doctors = list(preferences)
    slots = create_slots(capacities)
    rank_lookup = create_rank_lookup(preferences)
    cost_matrix = build_cost_matrix(doctors, slots, rank_lookup)
    slot_indices = hungarian_algorithm(cost_matrix)

    return {
        doctor: {
            "hospital": slots[slot_index]["hospital"],
            "slot": slots[slot_index]["slot"],
        }
        for doctor, slot_index in zip(doctors, slot_indices)
    }

Greedy baseline methods for comparison with the optimal assignment.

In [14]:
def _greedy_assignment(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
    doctor_order: list[str],
) -> dict[str, dict[str, str]]:
    remaining_capacity = capacities.copy()
    assignment = {}
    for doctor in doctor_order:
        for hospital in preferences[doctor]:
            if remaining_capacity[hospital] > 0:
                assignment[doctor] = {"hospital": hospital}
                remaining_capacity[hospital] -= 1
                break
    return assignment


def greedy_assignment(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
    doctor_order: list[str] | None = None,
) -> dict[str, dict[str, str]]:
    validate_inputs(preferences, capacities)
    order = list(preferences) if doctor_order is None else list(doctor_order)
    if set(order) != set(preferences) or len(order) != len(preferences):
        raise ValueError("Doctor order must contain every doctor exactly once.")
    return _greedy_assignment(preferences, capacities, order)


def run_randomized_greedy(
    preferences: dict[str, list[str]],
    capacities: dict[str, int],
    trials: int = 100,
    seed: int | None = 0,
) -> list[dict[str, dict[str, str]]]:
    if isinstance(trials, bool) or not isinstance(trials, int) or trials <= 0:
        raise ValueError("Trials must be a positive integer.")
    validate_inputs(preferences, capacities)
    generator = random.Random(seed)
    doctors = list(preferences)
    assignments = []
    for _ in range(trials):
        order = doctors.copy()
        generator.shuffle(order)
        assignments.append(_greedy_assignment(preferences, capacities, order))
    return assignments

Evaluation metrics for doctor-hospital assignments.

In [15]:
from collections import Counter
from statistics import mean
def evaluate_assignment(
    doctors: list[str],
    assignment: dict[str, dict[str, str]],
    preferences: dict[str, list[str]],
) -> dict[str, object]:
    """Evaluate a complete assignment using rank-based metrics."""
    if len(doctors) != len(set(doctors)):
        raise ValueError("Doctor list cannot contain duplicate names.")
    if set(doctors) != set(preferences):
        raise ValueError("Doctor list must match the preference dictionary.")
    if set(assignment) != set(doctors) or len(assignment) != len(doctors):
        raise ValueError("Every doctor must have exactly one assignment.")

    rank_lookup = create_rank_lookup(preferences)
    assigned_ranks = {}
    for doctor in doctors:
        hospital = assignment[doctor].get("hospital")
        if hospital not in rank_lookup[doctor]:
            raise ValueError(
                f"Assignment for {doctor!r} contains an unknown hospital."
            )
        assigned_ranks[doctor] = rank_lookup[doctor][hospital]
    ranks = list(assigned_ranks.values())
    total_cost = sum(ranks)
    first_choice_rate = sum(rank == 1 for rank in ranks)/len(doctors)
    top_three_count = sum(rank <= 3 for rank in ranks)

    return {
        "total_cost": total_cost,
        "average_rank": total_cost / len(doctors),
        "first_choice_rate": first_choice_rate,
        "top_three_rate": top_three_count / len(doctors),
        "worst_assigned_rank": max(ranks),
        "rank_distribution": dict(sorted(Counter(ranks).items())),
        "assigned_ranks": assigned_ranks,
    }


def summarize_trials(trial_metrics: list[dict[str, object]]) -> dict[str, float]:
    """Summarize repeated randomized greedy trials."""
    if not trial_metrics:
        raise ValueError("At least one trial is required.")
    costs = [float(metrics["total_cost"]) for metrics in trial_metrics]
    return {
        "mean_total_cost": mean(costs),
        "best_total_cost": min(costs),
        "worst_total_cost": max(costs),
        "mean_average_rank": mean(
            float(metrics["average_rank"]) for metrics in trial_metrics
        ),
        "mean_first_choice_rate": mean(
            float(metrics["first_choice_rate"]) for metrics in trial_metrics
        ),
        "mean_top_three_rate": mean(
            float(metrics["top_three_rate"]) for metrics in trial_metrics
        ),
        "mean_worst_assigned_rank": mean(
            float(metrics["worst_assigned_rank"]) for metrics in trial_metrics
        ),
    }
def main() -> None:
    hungarian_assignment = solve_hungarian(preferences, capacities)

    hungarian_metrics = evaluate_assignment(
        doctors, hungarian_assignment, preferences
    )

    greedy_assignments = run_randomized_greedy(
        preferences, capacities, trials=100, seed=0
    )

    greedy_metrics = [
        evaluate_assignment(doctors, assignment, preferences)
        for assignment in greedy_assignments
    ]

    print_assignment("Hungarian assignment", hungarian_assignment)
    print("\nHungarian metrics")
    print(hungarian_metrics)

    print("\nRandomized greedy summary (100 trials)")
    print(summarize_trials(greedy_metrics))

if __name__ == "__main__":
    main()



Hungarian assignment
--------------------------------------------------
D1 -> H3
D2 -> H2
D3 -> H4
D4 -> H1
D5 -> H8
D6 -> H6
D7 -> H10
D8 -> H9
D9 -> H8
D10 -> H9
D11 -> H8
D12 -> H1
D13 -> H6
D14 -> H2
D15 -> H5
D16 -> H3
D17 -> H8
D18 -> H5
D19 -> H1
D20 -> H9
D21 -> H4
D22 -> H10
D23 -> H10
D24 -> H8
D25 -> H3
D26 -> H5
D27 -> H1
D28 -> H10
D29 -> H7
D30 -> H10
D31 -> H5
D32 -> H1
D33 -> H4
D34 -> H6
D35 -> H7
D36 -> H5
D37 -> H4
D38 -> H7
D39 -> H9
D40 -> H2
D41 -> H2
D42 -> H3
D43 -> H7
D44 -> H9
D45 -> H5
D46 -> H10
D47 -> H8
D48 -> H6
D49 -> H1
D50 -> H3
D51 -> H7
D52 -> H9
D53 -> H4
D54 -> H2
D55 -> H4
D56 -> H3
D57 -> H6
D58 -> H2
D59 -> H7
D60 -> H6

Hungarian metrics
{'total_cost': 68, 'average_rank': 1.1333333333333333, 'first_choice_rate': 0.8666666666666667, 'top_three_rate': 1.0, 'worst_assigned_rank': 2, 'rank_distribution': {1: 52, 2: 8}, 'assigned_ranks': {'D1': 1, 'D2': 1, 'D3': 1, 'D4': 1, 'D5': 1, 'D6': 1, 'D7': 1, 'D8': 1, 'D9': 1, 'D10': 1, 'D11': 1, 'D12': 1, 